In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
from torch.utils.data import DataLoader, Dataset
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
device = 'cuda' if (torch.cuda.is_available()) else 'cpu'

In [ ]:
eps = 1e-6

def relu(x):
    return x.clamp(min = 0)

# def batch_norm(x, gamma):
#     batch_mean = x.mean(dim = 0)
#     batch_var = (x-batch_mean).var(dim = 0)
#     x_norm = (x - batch_mean) / torch.sqrt(batch_var + eps)
#     return gamma*x_norm

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
train_df, test_df = train_test_split(df, train_size = 0.8, random_state = 42)
train_df = train_df.to_numpy()
test_df = test_df.to_numpy()

y_train, x_train = train_df[:,0], train_df[:,1:].astype(np.float32)
y_train = pd.get_dummies(y_train).to_numpy().astype(np.float32)
y_test, x_test = test_df[:,0], test_df[:,1:].astype(np.float32)
y_test = pd.get_dummies(y_test).to_numpy().astype(np.float32)
# x_train, y_train, x_test, y_test = [t.to(device) for t in [torch.from_numpy(x_train), torch.from_numpy(y_train), torch.from_numpy(x_test), torch.from_numpy(y_test)]]

In [ ]:
print(y_train[0])

In [ ]:
batch_size = 64
num_workers = 4
epochs = 30
lr = 1e-2
hidden_features = 512

In [ ]:
class MNIST(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.n = len(x)
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        return self.x[idx],self.y[idx]

train_dataset = MNIST(x_train, y_train)
test_dataset = MNIST(x_test, y_test)

dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers = num_workers)

In [ ]:
def hidden_layer(input_size, output_size, activation):
    weights = torch.distributions.Uniform(-np.sqrt(1/input_size), np.sqrt(1/input_size)).sample((input_size, output_size)).to(device).requires_grad_()
    bias = torch.distributions.Uniform(-np.sqrt(1/input_size), np.sqrt(1/input_size)).sample((output_size,)).to(device).requires_grad_()
    return [weights, bias, activation]

layers = []
layers.append(hidden_layer(784, hidden_features, "relu"))
layers.append(hidden_layer(hidden_features, hidden_features, "relu"))
layers.append(hidden_layer(hidden_features, hidden_features, "relu"))
layers.append(hidden_layer(hidden_features, hidden_features, "relu"))
layers.append(hidden_layer(hidden_features, 10, "softmax"))

In [ ]:
layers[0][0]

In [ ]:
layers[1][1].shape

In [ ]:
y_train.shape

In [ ]:
# class sg_optim:
#     def __init__(self, params, lr = 1e-2):
#         self.params = params
#         self.lr = lr

#     def step(self):
#         for param in self.params:
#             param.data = param.data - self.lr * param.grad 

#     def zero_grad(self):
#         for param in self.params:
#             param.grad.zero_()

class sg_optim:
    def __init__(self, params, lr=1e-2):
        self.params = params
        self.lr = lr
    
    def step(self):
        for param in self.params:
            if param.grad is not None:
                param.data = param.data - self.lr * param.grad
    
    def zero_grad(self):
        for param in self.params:
            if param.grad is not None:
                param.grad.zero_()

In [ ]:
def cross_entropy(pred, target):
    loss_ = -torch.sum(target*torch.log(pred + 1e-8), dim = -1)
    return torch.mean(loss_)

In [ ]:
class Model:
    def __init__(self, layers):
        self.layers = layers
        self._params = self.params()
        self.optim = sg_optim(self._params)
    
    def forward(self, input_):
        output = input_ / 255.0
        for layer in self.layers:
            W, b, activation = layer
            output = output @ W + b
            if activation == "relu":
                output = relu(output)
            else:
                output = torch.softmax(output, dim=-1)
        return output
    
    def back_prop(self, input_, target):
        output = self.forward(input_)
        loss = cross_entropy(output, target)
        loss.backward()
        self.optim.step()
        self.optim.zero_grad()
        return loss
    
    def params(self):
        p = []
        for layer in self.layers:
            weight, bias, _ = layer
            p.append(weight)
            p.append(bias)
        return p


model = Model(layers)

In [ ]:
for epoch in range(epochs):
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)
        loss = model.back_prop(inputs, labels)
    if epoch%1 == 0:
        print(f"Epoch: {epoch}, Loss: {loss}")

In [ ]:
loss

In [ ]:
test_dataloader = DataLoader(test_dataset)
total_matches = 0
total_items = 0
for batch_idx, (inputs, labels) in enumerate(test_dataloader):
    inputs, labels = inputs.to(device), labels.to(device)
    output = model.forward(inputs)
    total_items += output.shape[0]
    total_matches += torch.sum(torch.sum(output * labels, dim = -1), dim = -1)
    # output_label = torch.argmax(output, dim = -1)
    # target_label = torch.argmax(labels, dim = -1)
    # total_matches += output_label.shape[0]
    # total_items += torch.sum(output_label )

print("Number of epochs run: ", epochs)
print("Accuracy: ", total_matches/total_items)